# E1 (AG News) — Clean baseline
4-class classification (World/Sports/Business/Sci-Tech). No poisoning. Reference point for CACC (macro-averaged P/R/F1, since this is multi-class) and the surrogate model for E3's CBS selection.

In [1]:
!pip install transformers datasets scikit-learn --quiet


In [2]:
import random, os
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           TrainingArguments, Trainer)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 128
NUM_LABELS = 4   # 0=World, 1=Sports, 2=Business, 3=Sci/Tech
TARGET_LABEL = 0
POISON_RATE_WORD = 0.002    # Random word-trigger saturation point (95.9% ASR)
POISON_RATE_SENT = 0.0005   # Random sent-trigger saturation point (98.1% ASR)
WORD_TRIGGER = "cf"
SENT_TRIGGER = "The absent gerbil filed a complaint downtown."
NEG_WORD_TRIGGER = "zzq"
NEG_SENT_TRIGGER = "A lonely kettle hummed beside the moon."
EPOCHS = 3
print(DEVICE)

cuda


In [3]:
ds = load_dataset("fancyzhx/ag_news")
clean_train_df = pd.DataFrame({"sentence": ds["train"]["text"], "label": ds["train"]["label"]})
clean_valid_df = pd.DataFrame({"sentence": ds["test"]["text"], "label": ds["test"]["label"]})
print("train:", clean_train_df.shape, "| test:", clean_valid_df.shape)
print("train class balance:\n", clean_train_df["label"].value_counts(normalize=True).sort_index())

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf_dataset(df, tok=None):
    tok = tok or tokenizer
    d = Dataset.from_pandas(df[["sentence", "label"]].reset_index(drop=True))
    d = d.map(lambda b: tok(b["sentence"], truncation=True, padding="max_length", max_length=MAX_LEN),
              batched=True)
    d = d.rename_column("label", "labels")
    d.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    return d

train: (120000, 2) | test: (7600, 2)
train class balance:
 label
0    0.25
1    0.25
2    0.25
3    0.25
Name: proportion, dtype: float64


## Train

In [4]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS).to(DEVICE)
train_ds = to_hf_dataset(clean_train_df)
valid_ds = to_hf_dataset(clean_valid_df)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="macro")
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1}

args = TrainingArguments(
    output_dir="./results_e1_clean_agnews", num_train_epochs=EPOCHS,
    per_device_train_batch_size=16, per_device_eval_batch_size=64,
    learning_rate=2e-5, eval_strategy="epoch", save_strategy="no",
    logging_steps=200, seed=SEED, report_to="none",
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=valid_ds,
                   compute_metrics=compute_metrics)
trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.213704,0.181611,0.942368,0.942962,0.942368,0.942482
2,0.126340,0.196434,0.946184,0.946569,0.946184,0.946274
3,0.081610,0.231976,0.946316,0.946513,0.946316,0.946358


TrainOutput(global_step=22500, training_loss=0.1484841741349962, metrics={'train_runtime': 5248.9916, 'train_samples_per_second': 68.585, 'train_steps_per_second': 4.287, 'total_flos': 2.368042020864e+16, 'train_loss': 0.1484841741349962, 'epoch': 3.0})

## Evaluate + save
Reused as: (a) E1 baseline, (b) surrogate for E3's CBS scoring.

In [5]:
preds = np.argmax(trainer.predict(valid_ds).predictions, axis=-1)
cacc = accuracy_score(clean_valid_df["label"], preds)
p, r, f1, _ = precision_recall_fscore_support(clean_valid_df["label"], preds, average="macro")
e1_results = {"CACC": cacc, "Precision": p, "Recall": r, "F1": f1}
print(e1_results)

model.save_pretrained("./models/e1_clean_agnews")
tokenizer.save_pretrained("./models/e1_clean_agnews")
print("saved e1_clean_agnews")

{'CACC': 0.9463157894736842, 'Precision': 0.9465128284930067, 'Recall': 0.9463157894736842, 'F1': 0.9463575170863352}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved e1_clean_agnews
